In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import mne
import numpy as np
import plotly.express as px
from mne_bids import (
    BIDSPath,
    read_raw_bids,
)

plt.switch_backend('Qt5Agg')
plt.ion()

import sys

sys.path.insert(0, str(Path.cwd().parent / 'code'))
from preprocessing import preprocess

In [ ]:
dataset = "ds003969"
tasks = ['think1', 'think2','med1breath', 'med2']

# Create a BIDS root directory in the MNE dataset folder
root = Path('/home/jackhoyle')
bids_root = root / 'mne_data' / dataset
bids_root.mkdir(parents=True, exist_ok=True)

In [ ]:
subject = "001"
task = 'think1'

In [ ]:
raw: mne.io.edf.edf.RawBDF = read_raw_bids(
    BIDSPath(subject=subject, root=bids_root, task=task, datatype="eeg"),
    verbose=True
)
raw.load_data()
assert type(raw) == mne.io.edf.edf.RawBDF

In [ ]:
raw.resample(256)

In [ ]:
fig = px.bar(np.var(raw.get_data(), axis=1), log_y=True)
fig.update_xaxes(tickvals=np.arange(len(raw.ch_names)), ticktext=raw.ch_names)

In [ ]:
orig_data = raw.get_data()[:,:1000].copy()

In [ ]:
raw.set_eeg_reference(['EXG6']) # EXG6 is right mastoid

In [ ]:
new_data = raw.get_data()[:,:1000].copy()

In [ ]:
px.imshow(new_data-orig_data)

In [ ]:
px.scatter(x=new_data[0,:], y=orig_data[0,:] - new_data[0,:])

In [ ]:
px.scatter(x=orig_data[0,:], y=new_data[0,:])

In [ ]:
px.scatter(x=orig_data[1,:], y=new_data[1,:])

In [ ]:
preprocess(raw, subsample_freq=256, # reference=['EXG6'], 
           low_pass_freq=1, high_pass_freq=None, bad_channel_variance_threshold=0.01)

In [ ]:
raw.set_eeg_reference('EXG6') # EXG6 is right mastoid

In [ ]:
raw.plot(duration=30, block=True)

In [ ]:
(picks := [x for x in raw.ch_names if 'F' in x])

In [ ]:
raw.compute_psd(picks=picks).plot();